# Retrospective Stage-2 Variant-A qualification operator

This notebook is an **unexecuted operator wrapper** around the merged FieldBridge commands. It is pinned to merge commit `d3f7c3b0123b76c187478c7775c40d33e574daf7` and the reviewed contracts `stage2-photometry-factorization-v1`, `stage2-photometry-continuity-reference-v2`, and `stage2-photometry-variant-a-qualification-v1`. It does not reproduce photometry, interpolation, VAE, metric, threshold, or cohort-classification arithmetic in notebook cells.

The only eligible data roles are retrospective `R/train` for fitting and the complete retrospective `R/validation` inventory for qualification. No endpoint selection or curated subset is accepted. Every `P` identity is classified and excluded before source hashing or array loading. Gate 0.1 is used only to build and hash-verify a separately labelled external continuity reference; it never selects qualification cases, supplies calibration targets, or enters Variant-A metric or threshold arithmetic. The notebook never builds a canonical latent bank, trains a model, evaluates private/prospective travellers, modifies the checkout, or enables descriptor coupling. Run cells in order and stop at both manual review gates.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/GuillermoTafoya/MRIxFields.git"
PINNED_COMMIT = "d3f7c3b0123b76c187478c7775c40d33e574daf7"
EXPECTED_PARENTS = (
    "fe2c878439f310e8e76ac9b871caae942a5e600f",
    "075f0bb6a640f8bbccc734993f4df43d6a0ff701",
)
REPO_DIR = Path("/content/MRIxFields-variant-a-d3f7c3b")

if REPO_DIR.exists():
    raise FileExistsError(
        f"Refusing to reuse or alter {REPO_DIR}. Start a fresh Colab runtime or remove it manually."
    )
subprocess.run(["git", "clone", "--no-checkout", REPOSITORY_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "fetch", "origin", PINNED_COMMIT], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", PINNED_COMMIT], cwd=REPO_DIR, check=True)

def git_text(*args: str) -> str:
    return subprocess.check_output(["git", *args], cwd=REPO_DIR, text=True).strip()

actual_head = git_text("rev-parse", "HEAD")
origin_main = git_text("rev-parse", "refs/remotes/origin/main")
parent_line = git_text("rev-list", "--parents", "-n", "1", "HEAD").split()
actual_parents = tuple(parent_line[1:])
status = git_text("status", "--porcelain")
detached = subprocess.run(
    ["git", "symbolic-ref", "-q", "HEAD"], cwd=REPO_DIR, capture_output=True
).returncode != 0
if actual_head != PINNED_COMMIT:
    raise RuntimeError(f"Detached HEAD mismatch: {actual_head}, expected {PINNED_COMMIT}.")
if actual_parents != EXPECTED_PARENTS:
    raise RuntimeError(f"Merge parents changed: {actual_parents} != {EXPECTED_PARENTS}.")
if status or not detached:
    raise RuntimeError(f"Checkout must be detached and clean; status={status!r}, detached={detached}.")
pinned_is_origin_main_ancestor = subprocess.run(
    ["git", "merge-base", "--is-ancestor", PINNED_COMMIT, "refs/remotes/origin/main"],
    cwd=REPO_DIR,
).returncode == 0
if not pinned_is_origin_main_ancestor:
    raise RuntimeError("Pinned reviewed commit is not an ancestor of the observed origin/main.")
print({"head": actual_head, "observed_origin_main": origin_main, "pinned_is_origin_main_ancestor": pinned_is_origin_main_ancestor, "parents": actual_parents, "detached": detached, "clean": True})

In [ ]:
import importlib
import json
import tomllib

# Install the base, evaluation, and official-evaluation requirements declared by this checkout.
with (REPO_DIR / "pyproject.toml").open("rb") as handle:
    project = tomllib.load(handle)["project"]
optional = project["optional-dependencies"]
requirements = list(dict.fromkeys(
    [*project["dependencies"], *optional["evaluation"], *optional["official-evaluation"]]
))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *requirements],
    check=True,
)

SOURCE_DIR = str(REPO_DIR / "src")
sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.environ["PYTHONPATH"] = SOURCE_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")
if SOURCE_DIR in sys.path:
    sys.path.remove(SOURCE_DIR)
sys.path.insert(0, SOURCE_DIR)
for module_name in tuple(sys.modules):
    if module_name == "fieldbridge" or module_name.startswith("fieldbridge."):
        del sys.modules[module_name]
importlib.invalidate_caches()
import fieldbridge

package_file = Path(fieldbridge.__file__).resolve()
if REPO_DIR not in package_file.parents:
    raise RuntimeError(f"FieldBridge import escaped the pinned checkout: {package_file}.")
CLI_ENV = os.environ.copy()
CLI_ENV["PYTHONPATH"] = SOURCE_DIR + os.pathsep + CLI_ENV.get("PYTHONPATH", "")
CLI_ENV["PYTHONDONTWRITEBYTECODE"] = "1"
probe = subprocess.check_output(
    [sys.executable, "-c", "import pathlib,fieldbridge; print(pathlib.Path(fieldbridge.__file__).resolve())"],
    cwd=REPO_DIR, env=CLI_ENV, text=True,
).strip()
if Path(probe) != package_file or git_text("status", "--porcelain"):
    raise RuntimeError("Dependency setup changed the checkout or subprocess imports are not pinned.")
print(json.dumps({"fieldbridge_import": str(package_file), "requirements": requirements}, indent=2))

## External operator inputs

Supply only external paths. The immutable split-v3 remains byte-identical scientific source provenance. Its reviewed Windows root `D:\MRI_Field_2026\Data` is mapped component-by-component to `RETROSPECTIVE_DATA_ROOT`; an atomic no-clobber operational split is written under `EXTERNAL_OUTPUT_ROOT` and used by the official fit and audit commands. Separate frozen inventory SHA-256 values are required for fitting and qualification. The qualification inventory is the complete eligible `R/validation` inventory after production cohort classification, with no endpoint or performance-based selection. Each inventory hash covers split, record and subject-group identity, domain, root-relative source path, path-identity hash, byte count, and source-file SHA-256. No `P` file is opened to construct either inventory.

In [ ]:
import re
from google.colab import drive

drive.mount("/content/drive")

FROZEN_SPLIT_V3_JSON = Path(input("External frozen split-v3 JSON: " ).strip()).expanduser()
RETROSPECTIVE_DATA_ROOT = Path(input("External retrospective data root: " ).strip()).expanduser()
FROZEN_STAGE1_VAE_CONFIG = Path(input("External frozen Stage-1 VAE config YAML: " ).strip()).expanduser()
FROZEN_VAE_CHECKPOINT = Path(input("External frozen VAE checkpoint: " ).strip()).expanduser()
GATE01_RESULT_JSON = Path(input("External reviewed Gate 0.1 result JSON: " ).strip()).expanduser()
EXTERNAL_OUTPUT_ROOT = Path(input("New external Variant-A qualification output root: " ).strip()).expanduser()
REVIEWED_WINDOWS_SOURCE_ROOT = r"D:\MRI_Field_2026\Data"

EXPECTED_SPLIT_V3_SHA256 = input("Expected split-v3 file SHA-256: " ).strip().lower()
EXPECTED_RETROSPECTIVE_FIT_INVENTORY_SHA256 = input("Expected complete R/train fit-inventory SHA-256: " ).strip().lower()
FROZEN_RETROSPECTIVE_QUALIFICATION_INVENTORY_SHA256 = input("Frozen complete R/validation qualification-inventory SHA-256: " ).strip().lower()
EXPECTED_VAE_CONFIG_SHA256 = input("Expected frozen VAE config SHA-256: " ).strip().lower()
EXPECTED_VAE_CHECKPOINT_SHA256 = input("Expected frozen VAE checkpoint SHA-256: " ).strip().lower()
EXPECTED_GATE01_RESULT_SHA256 = input("Expected Gate 0.1 result SHA-256: " ).strip().lower()
ALLOW_EXACT_RESUME = False  # Set True only to resume this exact, previously reviewed output root.

for label, value in {
    "split-v3": EXPECTED_SPLIT_V3_SHA256,
    "retrospective fit inventory": EXPECTED_RETROSPECTIVE_FIT_INVENTORY_SHA256,
    "retrospective qualification inventory": FROZEN_RETROSPECTIVE_QUALIFICATION_INVENTORY_SHA256,
    "VAE config": EXPECTED_VAE_CONFIG_SHA256,
    "VAE checkpoint": EXPECTED_VAE_CHECKPOINT_SHA256,
    "Gate 0.1 result": EXPECTED_GATE01_RESULT_SHA256,
}.items():
    if re.fullmatch(r"[0-9a-f]{64}", value) is None:
        raise ValueError(f"{label} requires an exact lowercase SHA-256.")
GATE01_CONTINUITY_REFERENCE_IDENTITY = (
    "gate01-external-continuity-only-source-sha256:" + EXPECTED_GATE01_RESULT_SHA256
)

In [ ]:
# Read-only input, cohort, role, checkpoint, runtime, and publication preflight.
from collections import Counter
from pathlib import PureWindowsPath
from unittest import mock
import copy
import hashlib
import shutil
import torch

import fieldbridge.cli as fb_cli
from fieldbridge.data.manifests import record_from_mapping
from fieldbridge.data.photometry_factorization import (
    all_photometry_domain_labels,
    assert_variant_a_external_path,
    capture_photometry_code_provenance,
    sha256_file,
    sha256_json,
    sha256_text,
    write_json_atomic,
)
from fieldbridge.data.vae_splits import (
    VaeSplits,
    load_vae_splits,
    vae_splits_fingerprint,
    vae_splits_recovery_fingerprint_v3,
)
from fieldbridge.evaluation.stage2_photometry_baseline import (
    build_continuity_reference_from_gate01,
)

if git_text("rev-parse", "HEAD") != PINNED_COMMIT or git_text("status", "--porcelain"):
    raise RuntimeError("Pinned checkout is no longer clean. Stop without processing data.")
file_inputs = {
    "split_v3": (FROZEN_SPLIT_V3_JSON, EXPECTED_SPLIT_V3_SHA256),
    "vae_config": (FROZEN_STAGE1_VAE_CONFIG, EXPECTED_VAE_CONFIG_SHA256),
    "vae_checkpoint": (FROZEN_VAE_CHECKPOINT, EXPECTED_VAE_CHECKPOINT_SHA256),
    "gate01_result": (GATE01_RESULT_JSON, EXPECTED_GATE01_RESULT_SHA256),
}
input_hashes = {}
resolved_inputs = {}
for label, (path, expected_hash) in file_inputs.items():
    resolved = assert_variant_a_external_path(path, repo_root=REPO_DIR)
    if not resolved.is_file():
        raise FileNotFoundError(f"Missing {label}: {resolved}")
    actual_hash = sha256_file(resolved)
    if actual_hash != expected_hash:
        raise RuntimeError(f"{label} SHA-256 mismatch: {actual_hash} != {expected_hash}.")
    input_hashes[label] = actual_hash
    resolved_inputs[label] = resolved
FROZEN_SPLIT_V3_JSON = resolved_inputs["split_v3"]
FROZEN_STAGE1_VAE_CONFIG = resolved_inputs["vae_config"]
FROZEN_VAE_CHECKPOINT = resolved_inputs["vae_checkpoint"]
GATE01_RESULT_JSON = resolved_inputs["gate01_result"]

data_root = assert_variant_a_external_path(RETROSPECTIVE_DATA_ROOT, repo_root=REPO_DIR)
output_root = assert_variant_a_external_path(EXTERNAL_OUTPUT_ROOT, repo_root=REPO_DIR)
if not data_root.is_dir():
    raise NotADirectoryError(f"Retrospective root does not exist: {data_root}")
for child, parent in ((output_root, data_root), (data_root, output_root)):
    try:
        child.relative_to(parent)
    except ValueError:
        continue
    raise ValueError("The output root and immutable retrospective data root must be separate trees.")

if output_root.exists() and not output_root.is_dir():
    raise NotADirectoryError(f"Output root is not a directory: {output_root}")
if output_root.exists() and any(output_root.iterdir()) and not ALLOW_EXACT_RESUME:
    raise FileExistsError("Use a new empty output root, or explicitly authorize exact resume.")
output_root.mkdir(parents=True, exist_ok=True)
if not output_root.is_dir():
    raise RuntimeError(f"Could not create external output root: {output_root}")
atomic_probe_path = output_root / f".variant-a-atomic-preflight-{os.getpid()}.json"
try:
    write_json_atomic(atomic_probe_path, {"probe": PINNED_COMMIT})
    if json.loads(atomic_probe_path.read_text(encoding="utf-8"))["probe"] != PINNED_COMMIT:
        raise RuntimeError("Atomic publication preflight returned changed content.")
finally:
    atomic_probe_path.unlink(missing_ok=True)

# Preserve and validate the immutable Windows-rooted scientific split first.
original_split_bytes = FROZEN_SPLIT_V3_JSON.read_bytes()
original_split_file_sha256 = hashlib.sha256(original_split_bytes).hexdigest()
if original_split_file_sha256 != EXPECTED_SPLIT_V3_SHA256:
    raise RuntimeError("Original split changed after initial input verification.")
original_splits = load_vae_splits(FROZEN_SPLIT_V3_JSON)
original_split_membership = vae_splits_fingerprint(original_splits)
original_split_recovery = vae_splits_recovery_fingerprint_v3(original_splits)
original_split_payload = json.loads(original_split_bytes.decode("utf-8"))
if not isinstance(original_split_payload, dict):
    raise ValueError("Original split-v3 root must be a JSON object.")

reviewed_old_root = PureWindowsPath(REVIEWED_WINDOWS_SOURCE_ROOT)
if not reviewed_old_root.is_absolute():
    raise ValueError("Reviewed Windows source root must be absolute.")
resolved_data_root = data_root.resolve(strict=True)

def remap_reviewed_windows_path(raw_path: str) -> Path:
    source = PureWindowsPath(str(raw_path))
    source_parts = tuple(part.casefold() for part in source.parts)
    root_parts = tuple(part.casefold() for part in reviewed_old_root.parts)
    if not source.is_absolute() or source_parts[:len(root_parts)] != root_parts:
        raise ValueError(
            f"Frozen split path is outside reviewed Windows root {reviewed_old_root}: {raw_path}"
        )
    relative_parts = source.parts[len(reviewed_old_root.parts):]
    if not relative_parts or any(part in {"", ".", ".."} for part in relative_parts):
        raise ValueError(f"Frozen split path has an unsafe relative suffix: {raw_path}")
    mapped = resolved_data_root.joinpath(*relative_parts).resolve(strict=False)
    try:
        mapped.relative_to(resolved_data_root)
    except ValueError as exc:
        raise ValueError(f"Remapped source escapes retrospective root: {mapped}") from exc
    return mapped

operational_split_payload = copy.deepcopy(original_split_payload)
mapping_entries = []
for split_name in ("train", "validation", "test"):
    records_payload = operational_split_payload.get("splits", {}).get(split_name)
    if not isinstance(records_payload, list):
        raise ValueError(f"Original split is missing record list {split_name!r}.")
    for record_payload in records_payload:
        if not isinstance(record_payload, dict) or "image_path" not in record_payload:
            raise ValueError(f"Malformed {split_name} record in original split.")
        old_path = str(record_payload["image_path"])
        new_path = remap_reviewed_windows_path(old_path)
        record_payload["image_path"] = str(new_path)
        mapping_entries.append({
            "split": split_name,
            "record_identity": str(record_payload.get("case_id", "")),
            "old_path": old_path,
            "new_path": str(new_path),
        })
mapping_entries.sort(key=lambda item: (item["split"], item["record_identity"], item["old_path"]))
mapping_identity_sha256 = sha256_json(mapping_entries)
operational_metadata = dict(operational_split_payload.get("metadata", {}))
operational_metadata["colab_path_remap"] = {
    "contract": "stage2-colab-windows-root-remap-v1",
    "original_split_file_sha256": original_split_file_sha256,
    "original_membership_fingerprint": original_split_membership,
    "original_recovery_fingerprint_v3": original_split_recovery,
    "reviewed_old_root": str(reviewed_old_root),
    "operational_new_root": str(resolved_data_root),
    "remapped_record_count": len(mapping_entries),
    "mapping_identity_sha256": mapping_identity_sha256,
}
operational_split_payload["metadata"] = operational_metadata

def records_from_operational_payload(split_name: str):
    return tuple(record_from_mapping(item) for item in operational_split_payload["splits"][split_name])

operational_candidate = VaeSplits(
    train=records_from_operational_payload("train"),
    validation=records_from_operational_payload("validation"),
    test=records_from_operational_payload("test"),
    seed=int(operational_split_payload["seed"]),
    fractions=tuple(float(value) for value in operational_split_payload["fractions"]),
    metadata=operational_metadata,
)
operational_split_membership = vae_splits_fingerprint(operational_candidate)
operational_split_recovery = vae_splits_recovery_fingerprint_v3(operational_candidate)
operational_split_payload["fingerprint"] = operational_split_membership
operational_split_payload["recovery_fingerprint_v3"] = operational_split_recovery
if operational_split_membership != original_split_membership:
    raise RuntimeError("Path remapping changed split membership.")
original_assignments = {
    name: tuple(sorted(record.case_id for record in original_splits.records_for(name)))
    for name in ("train", "validation", "test")
}
operational_assignments = {
    name: tuple(sorted(record.case_id for record in operational_candidate.records_for(name)))
    for name in ("train", "validation", "test")
}
if operational_assignments != original_assignments:
    raise RuntimeError("Path remapping changed train/validation/test assignments.")

OPERATIONAL_SPLIT_V3_JSON = output_root / "split_v3_colab_operational.json"
expected_operational_bytes = (
    json.dumps(operational_split_payload, indent=2, sort_keys=True, allow_nan=False) + "\n"
).encode("utf-8")
expected_operational_file_sha256 = hashlib.sha256(expected_operational_bytes).hexdigest()
if OPERATIONAL_SPLIT_V3_JSON.exists():
    if not ALLOW_EXACT_RESUME:
        raise FileExistsError("Operational split exists; exact resume was not authorized.")
    if OPERATIONAL_SPLIT_V3_JSON.read_bytes() != expected_operational_bytes:
        raise RuntimeError("Existing operational split does not exactly match deterministic remapping.")
else:
    write_json_atomic(OPERATIONAL_SPLIT_V3_JSON, operational_split_payload)
operational_split_file_sha256 = sha256_file(OPERATIONAL_SPLIT_V3_JSON)
if operational_split_file_sha256 != expected_operational_file_sha256:
    raise RuntimeError("Operational split file SHA-256 differs from deterministic bytes.")
splits = load_vae_splits(OPERATIONAL_SPLIT_V3_JSON)
split_membership = vae_splits_fingerprint(splits)
split_recovery = vae_splits_recovery_fingerprint_v3(splits)
if split_membership != operational_split_membership or split_recovery != operational_split_recovery:
    raise RuntimeError("Reloaded operational split fingerprints do not match sealed values.")
if {name: tuple(sorted(record.case_id for record in splits.records_for(name))) for name in ("train", "validation", "test")} != original_assignments:
    raise RuntimeError("Reloaded operational split changed an assignment.")

def assert_original_split_byte_immutable() -> None:
    if FROZEN_SPLIT_V3_JSON.read_bytes() != original_split_bytes:
        raise RuntimeError("Original frozen split bytes changed during operational preparation.")

assert_original_split_byte_immutable()
split_remap_provenance = {
    "original_split_path_identity_sha256": sha256_text(str(FROZEN_SPLIT_V3_JSON)),
    "original_split_file_sha256": original_split_file_sha256,
    "operational_split_path_identity_sha256": sha256_text(str(OPERATIONAL_SPLIT_V3_JSON)),
    "operational_split_file_sha256": operational_split_file_sha256,
    "original_membership_fingerprint": original_split_membership,
    "operational_membership_fingerprint": split_membership,
    "original_recovery_fingerprint_v3": original_split_recovery,
    "operational_recovery_fingerprint_v3": split_recovery,
    "reviewed_old_root": str(reviewed_old_root),
    "operational_new_root": str(resolved_data_root),
    "remapped_record_count": len(mapping_entries),
    "mapping_identity_sha256": mapping_identity_sha256,
    "membership_preserved": split_membership == original_split_membership,
    "all_split_assignments_preserved": operational_assignments == original_assignments,
    "original_file_remained_byte_identical": FROZEN_SPLIT_V3_JSON.read_bytes() == original_split_bytes,
}
array_load_attempts = 0
def forbidden_array_load(*args, **kwargs):
    global array_load_attempts
    array_load_attempts += 1
    raise AssertionError("Array loading is forbidden during cohort/role preflight.")
with mock.patch.object(fb_cli, "load_volume", side_effect=forbidden_array_load):
    fit_records, fit_excluded = fb_cli._select_variant_a_retrospective_records(
        splits.train, split="train"
    )
    qualification_records, qualification_excluded = fb_cli._select_variant_a_retrospective_records(
        splits.validation, split="validation"
    )
    classified = {
        split_name: tuple(fb_cli._classify_variant_a_split_record(record) for record in records)
        for split_name, records in {
            "train": splits.train, "validation": splits.validation, "test": splits.test
        }.items()
    }
if array_load_attempts != 0:
    raise AssertionError("Production cohort selection attempted to load an array.")

def identity_set(items):
    return {item.case_identity for item in items}
for split_name, selected, excluded in (
    ("train", fit_records, fit_excluded),
    ("validation", qualification_records, qualification_excluded),
):
    expected_r = {item.case_identity for item in classified[split_name] if item.cohort == "R"}
    expected_p = {item.case_identity for item in classified[split_name] if item.cohort == "P"}
    selected_ids = {str(record.case_id) for record in selected}
    excluded_ids = {str(item["record_identity"]) for item in excluded}
    if selected_ids != expected_r or excluded_ids != expected_p or selected_ids & excluded_ids:
        raise AssertionError(f"Fail-closed R/P selection proof failed for {split_name}.")
if not fit_records or not qualification_records:
    raise ValueError("Both R/train fitting and R/validation qualification require records.")
expected_domains = set(all_photometry_domain_labels())
fit_domain_counts = dict(sorted(Counter(record.domain.label for record in fit_records).items()))
qualification_domain_counts = dict(sorted(Counter(record.domain.label for record in qualification_records).items()))
if set(fit_domain_counts) != expected_domains or set(qualification_domain_counts) != expected_domains:
    raise ValueError("Preflight requires all 15 domains in both eligible roles.")

inventory_records = []
for split_name, records in (("train", fit_records), ("validation", qualification_records)):
    for record in records:
        cohort_identity = fb_cli._classify_variant_a_split_record(record)
        if cohort_identity.cohort != "R":
            raise AssertionError("Only production-classified R records may enter an inventory.")
        source = Path(record.image_path)
        if not source.is_absolute():
            raise ValueError(
                f"Frozen split source paths must be absolute because the official CLI uses them verbatim: {source}"
            )
        source = source.resolve(strict=True)
        try:
            relative = source.relative_to(data_root.resolve(strict=True))
        except ValueError as exc:
            raise ValueError(f"Eligible source escapes retrospective root: {source}") from exc
        inventory_records.append({
            "split": split_name,
            "record_identity": str(record.case_id),
            "record_identity_sha256": sha256_text(str(record.case_id)),
            "subject_group_identity": cohort_identity.subject_group_identity,
            "domain": record.domain.label,
            "relative_source_path": relative.as_posix(),
            "source_path_identity_sha256": sha256_text(str(record.image_path)),
            "source_bytes": source.stat().st_size,
            "source_file_sha256": sha256_file(source),
        })
inventory_records.sort(key=lambda item: (item["split"], item["domain"], item["record_identity"]))
fit_inventory_records = [item for item in inventory_records if item["split"] == "train"]
qualification_inventory_records = [
    item for item in inventory_records if item["split"] == "validation"
]
fit_inventory_sha256 = sha256_json(fit_inventory_records)
qualification_inventory_sha256 = sha256_json(qualification_inventory_records)
if fit_inventory_sha256 != EXPECTED_RETROSPECTIVE_FIT_INVENTORY_SHA256:
    raise RuntimeError(
        f"Complete R/train fit inventory mismatch: {fit_inventory_sha256} != "
        f"{EXPECTED_RETROSPECTIVE_FIT_INVENTORY_SHA256}."
    )
if qualification_inventory_sha256 != FROZEN_RETROSPECTIVE_QUALIFICATION_INVENTORY_SHA256:
    raise RuntimeError(
        f"Complete R/validation qualification inventory mismatch: "
        f"{qualification_inventory_sha256} != "
        f"{FROZEN_RETROSPECTIVE_QUALIFICATION_INVENTORY_SHA256}."
    )
qualification_record_identities = sorted(
    item["record_identity"] for item in qualification_inventory_records
)
qualification_all_classifier_r_identities = sorted(
    item.case_identity for item in classified["validation"] if item.cohort == "R"
)
if qualification_record_identities != qualification_all_classifier_r_identities:
    raise AssertionError("Qualification inventory is not the complete classified R/validation set.")
qualification_subject_group_counts = dict(sorted(Counter(
    item["subject_group_identity"] for item in qualification_inventory_records
).items()))
qualification_source_file_identities = [
    {
        "record_identity": item["record_identity"],
        "source_path_identity_sha256": item["source_path_identity_sha256"],
        "source_file_sha256": item["source_file_sha256"],
    }
    for item in qualification_inventory_records
]
qualification_inventory_evidence = {
    "inventory_sha256": qualification_inventory_sha256,
    "complete_eligible_record_count": len(qualification_inventory_records),
    "sorted_record_identities": qualification_record_identities,
    "sorted_record_identities_sha256": sha256_json(qualification_record_identities),
    "domain_counts": qualification_domain_counts,
    "all_15_domains_present": set(qualification_domain_counts) == expected_domains,
    "subject_group_counts": qualification_subject_group_counts,
    "unique_subject_group_count": len(qualification_subject_group_counts),
    "source_file_identities": qualification_source_file_identities,
    "source_file_identities_sha256": sha256_json(qualification_source_file_identities),
    "excluded_P_identities_and_reasons": list(qualification_excluded),
    "excluded_P_identities_and_reasons_sha256": sha256_json(list(qualification_excluded)),
    "exclusion_before_array_loading": array_load_attempts == 0,
    "prospective_source_files_opened": 0,
    "complete_classifier_R_validation_set": True,
    "endpoint_selection_used": False,
    "curated_subset_used": False,
    "performance_based_selection_used": False,
    "selection_inputs": ["frozen split-v3 role", "production R/P classifier"],
    "performance_or_target_inputs_read_for_selection": [],
}

# Gate 0.1 is validated only as the source of a separately labelled continuity reference.
gate01_payload = json.loads(GATE01_RESULT_JSON.read_text(encoding="utf-8"))
continuity_probe = build_continuity_reference_from_gate01(
    gate01_payload,
    source_result_sha256=input_hashes["gate01_result"],
    evaluation_identity=GATE01_CONTINUITY_REFERENCE_IDENTITY,
)

# Strictly construct the frozen VAE and load its state on CPU; perform no inference.
vae_config = fb_cli.load_yaml_config(FROZEN_STAGE1_VAE_CONFIG)
model_config = fb_cli._model_config(vae_config)
encoder_probe = fb_cli.build_encoder("kl_vae", **fb_cli._kl_vae_kwargs(model_config, "encoder"))
decoder_probe = fb_cli.build_decoder("kl_vae", **fb_cli._kl_vae_kwargs(model_config, "decoder"))
checkpoint_probe = fb_cli.load_checkpoint(FROZEN_VAE_CHECKPOINT)
encoder_probe.load_state_dict(checkpoint_probe["encoder"], strict=True)
decoder_probe.load_state_dict(checkpoint_probe["decoder"], strict=True)
del checkpoint_probe, encoder_probe, decoder_probe
if not torch.cuda.is_available():
    raise RuntimeError("Official Variant-A qualification requires a CUDA Colab runtime.")
import nibabel, skimage, lpips, matplotlib  # noqa: F401 -- dependency preflight only

preflight_report = {
    "report_type": "variant-a-colab-operator-preflight-not-a-scientific-artifact",
    "code_commit": PINNED_COMMIT,
    "code_parents": list(EXPECTED_PARENTS),
    "observed_origin_main_sha": origin_main,
    "pinned_commit_is_origin_main_ancestor": pinned_is_origin_main_ancestor,
    "code_provenance": capture_photometry_code_provenance(REPO_DIR),
    "authoritative_contracts": [
        "stage2-photometry-factorization-v1",
        "stage2-photometry-continuity-reference-v2",
        "stage2-photometry-variant-a-qualification-v1",
    ],
    "input_file_sha256": input_hashes,
    "immutable_scientific_split_provenance": {
        "path_identity_sha256": split_remap_provenance["original_split_path_identity_sha256"],
        "file_sha256": original_split_file_sha256,
        "membership_fingerprint": original_split_membership,
        "recovery_fingerprint_v3": original_split_recovery,
        "remained_byte_identical": split_remap_provenance["original_file_remained_byte_identical"],
    },
    "operational_split_remap_provenance": split_remap_provenance,
    "retrospective_root_path_identity_sha256": sha256_text(str(data_root.resolve())),
    "retrospective_fit_inventory_sha256": fit_inventory_sha256,
    "retrospective_qualification_inventory_sha256": qualification_inventory_sha256,
    "selected_retrospective_source_bytes": sum(item["source_bytes"] for item in inventory_records),
    "retrospective_fit_inventory": fit_inventory_records,
    "retrospective_qualification_inventory_evidence": qualification_inventory_evidence,
    "split_membership_fingerprint": split_membership,
    "split_recovery_fingerprint_v3": split_recovery,
    "role_proof": {
        "fit": {"cohort": "R", "split": "train", "accepted": len(fit_records), "excluded_P": list(fit_excluded), "domain_counts": fit_domain_counts},
        "qualification": {"cohort": "R", "split": "validation", "accepted": len(qualification_records), "excluded_P": list(qualification_excluded), "domain_counts": qualification_domain_counts},
        "test_not_selected": {
            "record_count": len(splits.test),
            "R_count": sum(item.cohort == "R" for item in classified["test"]),
            "P_count": sum(item.cohort == "P" for item in classified["test"]),
        },
        "selection_array_load_attempts": array_load_attempts,
        "prospective_source_files_opened": 0,
        "qualification_uses_complete_R_validation_inventory": True,
        "performance_based_selection_used": False,
    },
    "vae_validation": {
        "config_sha256": input_hashes["vae_config"],
        "checkpoint_sha256": input_hashes["vae_checkpoint"],
        "strict_state_load": True,
        "inference_performed": False,
    },
    "gate01_use_boundary": {
        "continuity_reference_only": True,
        "continuity_identity": GATE01_CONTINUITY_REFERENCE_IDENTITY,
        "continuity_preview_sha256": continuity_probe.artifact_sha256,
        "qualification_case_selection_input": False,
        "calibration_target_input": False,
        "metric_or_threshold_arithmetic_input": False,
        "qualification_cli_gate01_argument_use": "continuity source-file SHA-256 verification only",
    },
    "output_root_path_identity_sha256": sha256_text(str(output_root.resolve())),
}
preflight_report["preflight_sha256"] = sha256_json(preflight_report)
PREFLIGHT_JSON = output_root / "variant_a_operator_preflight.json"
if PREFLIGHT_JSON.exists():
    if not ALLOW_EXACT_RESUME:
        raise FileExistsError("Preflight already exists; set ALLOW_EXACT_RESUME only for an exact restart.")
    prior_preflight = json.loads(PREFLIGHT_JSON.read_text(encoding="utf-8"))
    if prior_preflight != preflight_report:
        raise RuntimeError("Existing preflight is not an exact match for the current inputs.")
else:
    write_json_atomic(PREFLIGHT_JSON, preflight_report)
disk = shutil.disk_usage(output_root)
print(json.dumps({
    "preflight_json": str(PREFLIGHT_JSON),
    "preflight_sha256": preflight_report["preflight_sha256"],
    "original_split_file_sha256": original_split_file_sha256,
    "operational_split_file_sha256": operational_split_file_sha256,
    "original_split_membership": original_split_membership,
    "operational_split_membership": split_membership,
    "original_split_recovery_v3": original_split_recovery,
    "operational_split_recovery_v3": split_recovery,
    "remapped_record_count": len(mapping_entries),
    "mapping_identity_sha256": mapping_identity_sha256,
    "original_split_remained_byte_identical": split_remap_provenance["original_file_remained_byte_identical"],
    "fit_inventory_sha256": fit_inventory_sha256,
    "qualification_inventory_sha256": qualification_inventory_sha256,
    "qualification_record_identities_sha256": qualification_inventory_evidence["sorted_record_identities_sha256"],
    "fit_R_train": len(fit_records),
    "qualification_R_validation": len(qualification_records),
    "excluded_P_train": len(fit_excluded),
    "excluded_P_validation": len(qualification_excluded),
    "all_15_domains_fit": len(fit_domain_counts) == 15,
    "all_15_domains_qualification": len(qualification_domain_counts) == 15,
    "prospective_source_files_opened": 0,
    "selected_retrospective_source_bytes": preflight_report["selected_retrospective_source_bytes"],
    "cuda_device": torch.cuda.get_device_name(0),
    "output_free_bytes": disk.free,
}, indent=2))
print("STOP: send the external preflight JSON and inventory hash for review. Do not run the fit authorization cell yet.")

## Intentional review stop 1 — preflight

Stop here. External review must confirm the pinned code, input hashes, complete 15-domain `R/train` and `R/validation` role proof, all `P` exclusions, strict frozen-VAE load, output storage, and zero array-load attempts during selection.

In [ ]:
AUTHORIZE_FIT_AFTER_EXTERNAL_PREFLIGHT_REVIEW = False
if AUTHORIZE_FIT_AFTER_EXTERNAL_PREFLIGHT_REVIEW is not True:
    raise PermissionError("Manual authorization required after external preflight review.")

In [ ]:
from fieldbridge.data.photometry_factorization import FrozenPhotometryArtifact

VARIANT_A_CONFIG = REPO_DIR / "configs/experiment/stage2_photometry_factorization_a_v1.yaml"
PHOTOMETRY_ARTIFACT_JSON = output_root / "stage2_photometry_factorization_v1.json"
FIT_DIAGNOSTIC_LOG = output_root / "stage2_photometry_fit_diagnostic.log"
if git_text("rev-parse", "HEAD") != PINNED_COMMIT or git_text("status", "--porcelain"):
    raise RuntimeError("Checkout changed after preflight; fitting is forbidden.")
assert_original_split_byte_immutable()

def run_cli_with_visible_append_only_log(command, log_path: Path, operation: str) -> None:
    if log_path.exists():
        raise FileExistsError(f"Refusing to replace existing diagnostic log: {log_path}")
    descriptor = os.open(log_path, os.O_WRONLY | os.O_CREAT | os.O_EXCL | os.O_APPEND, 0o600)
    return_code = None
    with os.fdopen(descriptor, "w", encoding="utf-8", buffering=1) as diagnostic_log:
        diagnostic_log.write(json.dumps({"operation": operation, "code_commit": PINNED_COMMIT, "command": command}) + "\n")
        process = subprocess.Popen(
            command,
            cwd=REPO_DIR,
            env=CLI_ENV,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        if process.stdout is None:
            raise RuntimeError(f"{operation} did not expose a combined output stream.")
        for line in process.stdout:
            print(line, end="", flush=True)
            diagnostic_log.write(line)
            diagnostic_log.flush()
        return_code = process.wait()
        diagnostic_log.write(json.dumps({"operation": operation, "return_code": return_code}) + "\n")
        diagnostic_log.flush()
        os.fsync(diagnostic_log.fileno())
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
fit_command = [
    sys.executable, "-m", "fieldbridge.cli", "fit-stage2-photometry",
    "--config", str(VARIANT_A_CONFIG),
    "--split-json", str(OPERATIONAL_SPLIT_V3_JSON),
    "--out", str(PHOTOMETRY_ARTIFACT_JSON),
    "--device", "cpu",
    "--log-every", "10",
]
if PHOTOMETRY_ARTIFACT_JSON.exists():
    if not ALLOW_EXACT_RESUME:
        raise FileExistsError("Photometry artifact exists; exact resume was not authorized.")
    if not FIT_DIAGNOSTIC_LOG.is_file() or FIT_DIAGNOSTIC_LOG.stat().st_size == 0:
        raise RuntimeError("Exact-resume fit artifact is missing its append-only diagnostic log.")
    print("Exact resume: validating the completed no-clobber photometry artifact.")
else:
    run_cli_with_visible_append_only_log(fit_command, FIT_DIAGNOSTIC_LOG, "fit-stage2-photometry")
artifact = FrozenPhotometryArtifact.load(
    PHOTOMETRY_ARTIFACT_JSON,
    expected_split_file_sha256=operational_split_file_sha256,
    expected_membership_fingerprint=split_membership,
    expected_recovery_fingerprint=split_recovery,
)
assert_original_split_byte_immutable()
resolved_variant_config = fb_cli._load_variant_a_config(VARIANT_A_CONFIG)
if artifact.provenance["code_commit"] != PINNED_COMMIT:
    raise RuntimeError("Photometry artifact was not fitted by the pinned merge commit.")
if artifact.provenance["code_provenance"] != capture_photometry_code_provenance(REPO_DIR):
    raise RuntimeError("Photometry artifact code provenance does not match this clean checkout.")
if artifact.provenance["resolved_config_sha256"] != sha256_json(resolved_variant_config):
    raise RuntimeError("Photometry artifact config identity mismatch.")
expected_fit_content = {
    (item["record_identity"], item["domain"], item["source_file_sha256"])
    for item in inventory_records if item["split"] == "train"
}
actual_fit_content = {
    (item["record_identity"], item["domain"], item["source_file_sha256"])
    for item in artifact.provenance["accepted_records"]
}
if actual_fit_content != expected_fit_content:
    raise RuntimeError("Fitted artifact accepted-record content differs from preflight.")
fit_review = {
    "artifact_path": str(PHOTOMETRY_ARTIFACT_JSON),
    "artifact_file_sha256": sha256_file(PHOTOMETRY_ARTIFACT_JSON),
    "artifact_identity": artifact.artifact_sha256,
    "diagnostic_log": {"path": str(FIT_DIAGNOSTIC_LOG), "sha256": sha256_file(FIT_DIAGNOSTIC_LOG)},
    "contract_version": artifact.to_dict()["contract_version"],
    "source_split_file_sha256": artifact.provenance["source_split_file_sha256"],
    "source_membership_fingerprint": artifact.provenance["source_membership_fingerprint"],
    "source_recovery_fingerprint": artifact.provenance["source_recovery_fingerprint"],
    "domain_volume_counts": artifact.provenance["domain_volume_counts"],
    "field_and_volume_weights": artifact.provenance["weighting"],
    "accepted_records": artifact.provenance["accepted_records"],
    "accepted_records_sha256": artifact.provenance["accepted_records_sha256"],
    "excluded_prospective_records": artifact.provenance["excluded_prospective_records"],
    "excluded_prospective_records_sha256": artifact.provenance["excluded_prospective_records_sha256"],
    "eligibility_proof": artifact.provenance["eligibility_proof"],
}
print(json.dumps(fit_review, indent=2, sort_keys=True))
print("STOP: externally review the fitted artifact identity, counts, weights, records, exclusions, and hashes.")

## Intentional review stop 2 — fitted Variant-A artifact

Stop here. Review the immutable artifact and printed evidence before authorizing continuity construction or held-out qualification. The next phase still does not build a latent bank.

In [ ]:
AUTHORIZE_QUALIFICATION_AFTER_EXTERNAL_FIT_REVIEW = False
if AUTHORIZE_QUALIFICATION_AFTER_EXTERNAL_FIT_REVIEW is not True:
    raise PermissionError("Manual authorization required after external fit-artifact review.")

In [ ]:
from fieldbridge.evaluation.stage2_photometry_baseline import ContinuityReference

CONTINUITY_REFERENCE_JSON = output_root / "stage2_photometry_continuity_reference_v2.json"
if git_text("rev-parse", "HEAD") != PINNED_COMMIT or git_text("status", "--porcelain"):
    raise RuntimeError("Checkout changed after fit review; continuity construction is forbidden.")
continuity_command = [
    sys.executable, "-m", "fieldbridge.cli", "build-stage2-photometry-continuity-reference",
    "--gate01-result", str(GATE01_RESULT_JSON),
    "--evaluation-id", GATE01_CONTINUITY_REFERENCE_IDENTITY,
    "--out", str(CONTINUITY_REFERENCE_JSON),
]
if CONTINUITY_REFERENCE_JSON.exists():
    if not ALLOW_EXACT_RESUME:
        raise FileExistsError("Continuity reference exists; exact resume was not authorized.")
else:
    built_continuity = subprocess.run(
        continuity_command, cwd=REPO_DIR, env=CLI_ENV, check=True, capture_output=True, text=True
    )
    print(built_continuity.stdout)
continuity = ContinuityReference.load(CONTINUITY_REFERENCE_JSON, source_result_path=GATE01_RESULT_JSON)
if continuity.evaluation_identity != GATE01_CONTINUITY_REFERENCE_IDENTITY:
    raise RuntimeError("Continuity-only source identity mismatch.")
if continuity.artifact_sha256 != continuity_probe.artifact_sha256:
    raise RuntimeError("Published continuity reference differs from the preflight production preview.")
print(json.dumps({
    "path": str(CONTINUITY_REFERENCE_JSON),
    "file_sha256": sha256_file(CONTINUITY_REFERENCE_JSON),
    "artifact_sha256": continuity.artifact_sha256,
    "source_result_sha256": continuity.source_result_sha256,
    "evaluation_identity": continuity.evaluation_identity,
}, indent=2))

In [ ]:
QUALIFICATION_JSON = output_root / "stage2_photometry_variant_a_qualification_v1.json"
QUALIFICATION_DIAGNOSTIC_LOG = output_root / "stage2_photometry_qualification_diagnostic.log"
if git_text("rev-parse", "HEAD") != PINNED_COMMIT or git_text("status", "--porcelain"):
    raise RuntimeError("Checkout changed before qualification; stop without loading validation arrays.")
assert_original_split_byte_immutable()
# The Gate 0.1 path below is required only to re-hash the continuity source file.
qualification_command = [
    sys.executable, "-m", "fieldbridge.cli", "audit-stage2-photometry",
    "--config", str(VARIANT_A_CONFIG),
    "--split-json", str(OPERATIONAL_SPLIT_V3_JSON),
    "--artifact", str(PHOTOMETRY_ARTIFACT_JSON),
    "--vae-config", str(FROZEN_STAGE1_VAE_CONFIG),
    "--vae-checkpoint", str(FROZEN_VAE_CHECKPOINT),
    "--continuity-reference", str(CONTINUITY_REFERENCE_JSON),
    "--gate01-result", str(GATE01_RESULT_JSON),
    "--out", str(QUALIFICATION_JSON),
    "--device", "cuda",
    "--precision", "float32",
    "--log-every", "1",
]
if QUALIFICATION_JSON.exists():
    if not ALLOW_EXACT_RESUME:
        raise FileExistsError("Qualification result exists; exact resume was not authorized.")
    if not QUALIFICATION_DIAGNOSTIC_LOG.is_file() or QUALIFICATION_DIAGNOSTIC_LOG.stat().st_size == 0:
        raise RuntimeError("Exact-resume qualification result is missing its append-only diagnostic log.")
    print("Exact resume: validating the completed no-clobber qualification JSON.")
else:
    run_cli_with_visible_append_only_log(
        qualification_command, QUALIFICATION_DIAGNOSTIC_LOG, "audit-stage2-photometry"
    )
qualification = json.loads(QUALIFICATION_JSON.read_text(encoding="utf-8"))
assert_original_split_byte_immutable()
stored_result_sha256 = qualification.get("result_sha256")
qualification_without_hash = dict(qualification)
qualification_without_hash.pop("result_sha256", None)
if stored_result_sha256 != sha256_json(qualification_without_hash):
    raise RuntimeError("Qualification result content hash mismatch.")
if qualification.get("contract_version") != "stage2-photometry-variant-a-qualification-v1":
    raise RuntimeError("Unexpected qualification contract.")
if qualification.get("artifact_sha256") != artifact.artifact_sha256:
    raise RuntimeError("Qualification used a different photometry artifact.")
source_split = qualification.get("source_split", {})
if source_split != {
    "file_sha256": operational_split_file_sha256,
    "membership_fingerprint": split_membership,
    "recovery_fingerprint": split_recovery,
}:
    raise RuntimeError("Qualification split identity mismatch.")
vae_provenance = qualification.get("vae_provenance", {})
if vae_provenance.get("config_file_sha256") != input_hashes["vae_config"] or vae_provenance.get("checkpoint_sha256") != input_hashes["vae_checkpoint"]:
    raise RuntimeError("Qualification VAE identity mismatch.")
if vae_provenance.get("encoder_statistic") != "posterior_mean" or vae_provenance.get("encode_strategy") != "full" or vae_provenance.get("decode_strategy") != "full":
    raise RuntimeError("Qualification did not use full frozen posterior-mean VAE arithmetic.")
if qualification.get("resolved_config_sha256") != sha256_json(resolved_variant_config):
    raise RuntimeError("Qualification config identity mismatch.")
eligibility = qualification.get("eligibility_proof", {})
if not eligibility.get("all_cohort_R") or not eligibility.get("all_split_validation") or eligibility.get("prospective_accepted_count") != 0:
    raise RuntimeError("Qualification eligibility proof is not retrospective R/validation only.")
expected_validation_content = {
    (item["record_identity"], item["domain"], item["source_file_sha256"])
    for item in inventory_records if item["split"] == "validation"
}
actual_validation_content = {
    (item["record_identity"], item["domain"], item["source_file_sha256"])
    for item in qualification.get("records", [])
}
if actual_validation_content != expected_validation_content:
    raise RuntimeError("Qualification record/content identities differ from preflight.")
if sorted(item["record_identity"] for item in qualification.get("records", [])) != qualification_record_identities:
    raise RuntimeError("Qualification did not consume the complete frozen R/validation inventory.")
if qualification.get("record_count") != qualification_inventory_evidence["complete_eligible_record_count"]:
    raise RuntimeError("Qualification eligible-record count differs from the frozen inventory.")
if qualification.get("excluded_prospective_records") != list(qualification_excluded):
    raise RuntimeError("Qualification P exclusions differ from the pre-array-load proof.")
if set(qualification.get("aggregate", {}).get("per_domain", {})) != expected_domains:
    raise RuntimeError("Qualification evidence does not cover all 15 domains.")
continuity_evidence = qualification.get("stage1_reconstruction_ceiling_continuity", {})
if continuity_evidence.get("continuity_reference_sha256") != continuity.artifact_sha256:
    raise RuntimeError("Qualification continuity identity mismatch.")
if "not an additional qualification threshold" not in continuity_evidence.get("interpretation", ""):
    raise RuntimeError("Gate 0.1 continuity evidence is not separated from qualification thresholds.")

In [ ]:
# Render only already-computed official evidence; do not recompute scientific metrics.
assert_original_split_byte_immutable()
from io import BytesIO
import hashlib
import matplotlib.pyplot as plt

aggregate = qualification["aggregate"]
per_domain = aggregate["per_domain"]
domain_labels = sorted(per_domain)
photometry_checks = aggregate["photometry_checks"]
vae_checks = aggregate["vae_checks"]
all_thresholds_pass = (
    all(bool(value) for value in photometry_checks.values())
    and all(bool(value) for value in vae_checks.values())
    and bool(aggregate["photometry_factorization_pass"])
    and bool(aggregate["canonical_vae_compatibility_pass"])
    and not qualification["failure_classification"]
)

def status_table(values):
    return "\n".join(f"| `{key}` | {'PASS' if value else 'FAIL'} |" for key, value in sorted(values.items()))

domain_rows = []
for label in domain_labels:
    item = per_domain[label]
    domain_rows.append(
        f"| `{label}` | {item['count']} | {item['direct_roundtrip_nrmse']:.8g} | "
        f"{item['canonical_histogram_distance']:.8g} | {item['spearman']:.8g} | "
        f"{item['scaling_histogram_distance_worst']:.8g} | {item['scaling_ssim_worst']:.8g} |"
    )
review_markdown = f"""# Variant-A retrospective qualification review

- Overall threshold status: **{'PASS' if all_thresholds_pass else 'FAIL'}**
- Qualification contract: `{qualification['contract_version']}`
- Qualification result identity: `{qualification['result_sha256']}`
- Photometry artifact identity: `{artifact.artifact_sha256}`
- Immutable original split SHA-256: `{original_split_file_sha256}`
- Operational Colab split SHA-256: `{operational_split_file_sha256}`
- Path-remapping identity SHA-256: `{mapping_identity_sha256}`
- Complete R/validation inventory SHA-256: `{qualification_inventory_sha256}`
- Complete eligible R/validation records: `{qualification_inventory_evidence['complete_eligible_record_count']}`
- Sorted qualification record-identities SHA-256: `{qualification_inventory_evidence['sorted_record_identities_sha256']}`
- Frozen VAE config SHA-256: `{input_hashes['vae_config']}`
- Frozen VAE checkpoint SHA-256: `{input_hashes['vae_checkpoint']}`
- Gate 0.1 SHA-256: `{input_hashes['gate01_result']}`
- Continuity artifact identity: `{continuity.artifact_sha256}`
- Eligible records: `{qualification['record_count']}` retrospective `R/validation`
- Excluded prospective records: `{len(qualification['excluded_prospective_records'])}`
- Failure classification: `{qualification['failure_classification']}`

Qualification consumed the complete production-classified retrospective `R/validation` inventory. No endpoint, curated-subset, or performance-based selection was used. Gate 0.1 contributes only the separately labelled external continuity comparison and is not a calibration target or qualification threshold input. This report displays official result values only; it does not recompute metrics or thresholds. No latent bank was built, and descriptor coupling remains disabled.

## Complete qualification-inventory evidence

```json
{json.dumps(qualification_inventory_evidence, indent=2, sort_keys=True)}
```

## Immutable-source and operational-split provenance

```json
{json.dumps(split_remap_provenance, indent=2, sort_keys=True)}
```

## Photometry checks

| Check | Status |
|---|---|
{status_table(photometry_checks)}

## Canonical-VAE distribution-shift checks

| Check | Status |
|---|---|
{status_table(vae_checks)}

## Per-domain evidence

| Domain | n | Round-trip nRMSE | Histogram distance | Spearman | Scaling histogram worst | Scaling SSIM worst |
|---|---:|---:|---:|---:|---:|---:|
{chr(10).join(domain_rows)}

## Official macro evidence

```json
{json.dumps(aggregate['macro'], indent=2, sort_keys=True)}
```

## Official worst-domain evidence

```json
{json.dumps(aggregate['worst_domain'], indent=2, sort_keys=True)}
```

## Realized interpolation and monotonicity evidence

```json
{json.dumps(qualification['interpolation_qualification'], indent=2, sort_keys=True)}
```

## Grouped contrast-control evidence

```json
{json.dumps(qualification['contrast_preservation_control'], indent=2, sort_keys=True)}
```

## External Stage-1 continuity evidence

```json
{json.dumps(qualification['stage1_reconstruction_ceiling_continuity'], indent=2, sort_keys=True)}
```
"""

def publish_exact_bytes(path: Path, data: bytes) -> None:
    digest = hashlib.sha256(data).hexdigest()
    if path.exists():
        if not ALLOW_EXACT_RESUME or sha256_file(path) != digest:
            raise FileExistsError(f"Refusing to replace non-identical review output: {path}")
        return
    temporary = path.with_name(f".{path.name}.{os.getpid()}.tmp")
    try:
        with temporary.open("xb") as handle:
            handle.write(data)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    finally:
        temporary.unlink(missing_ok=True)

REVIEW_MARKDOWN = output_root / "stage2_photometry_variant_a_qualification_review.md"
publish_exact_bytes(REVIEW_MARKDOWN, review_markdown.encode("utf-8"))

fig, axes = plt.subplots(2, 1, figsize=(11, 6), constrained_layout=True)
x = list(range(len(domain_labels)))
axes[0].bar(x, [per_domain[label]["direct_roundtrip_nrmse"] for label in domain_labels])
axes[0].set_ylabel("direct round-trip nRMSE")
axes[0].set_title("Official Variant-A per-domain photometry evidence")
axes[1].bar(x, [per_domain[label]["canonical_histogram_distance"] for label in domain_labels])
axes[1].set_ylabel("canonical histogram distance")
axes[1].set_xticks(x, domain_labels, rotation=55, ha="right")
figure_buffer = BytesIO()
fig.savefig(figure_buffer, format="png", dpi=150)
plt.close(fig)
PHOTOMETRY_FIGURE = output_root / "stage2_photometry_variant_a_photometry_evidence.png"
publish_exact_bytes(PHOTOMETRY_FIGURE, figure_buffer.getvalue())

delta_keys = ("nrmse_absolute_increase", "nrmse_relative_increase", "ssim_decrease", "lpips_increase")
fig, axes = plt.subplots(2, 2, figsize=(11, 7), constrained_layout=True)
for axis, key in zip(axes.flat, delta_keys):
    axis.bar(x, [per_domain[label]["vae_deltas"][key] for label in domain_labels])
    axis.set_title(key.replace("_", " "))
    axis.set_xticks(x, domain_labels, rotation=70, ha="right", fontsize=7)
fig.suptitle("Official canonical-VAE per-domain distribution-shift evidence")
figure_buffer = BytesIO()
fig.savefig(figure_buffer, format="png", dpi=150)
plt.close(fig)
VAE_FIGURE = output_root / "stage2_photometry_variant_a_vae_evidence.png"
publish_exact_bytes(VAE_FIGURE, figure_buffer.getvalue())

review_outputs = {
    "operational_split_json": {"path": str(OPERATIONAL_SPLIT_V3_JSON), "sha256": operational_split_file_sha256},
    "split_mapping_identity_sha256": mapping_identity_sha256,
    "fit_diagnostic_log": {"path": str(FIT_DIAGNOSTIC_LOG), "sha256": sha256_file(FIT_DIAGNOSTIC_LOG)},
    "qualification_diagnostic_log": {"path": str(QUALIFICATION_DIAGNOSTIC_LOG), "sha256": sha256_file(QUALIFICATION_DIAGNOSTIC_LOG)},
    "qualification_json": {"path": str(QUALIFICATION_JSON), "sha256": sha256_file(QUALIFICATION_JSON)},
    "review_markdown": {"path": str(REVIEW_MARKDOWN), "sha256": sha256_file(REVIEW_MARKDOWN)},
    "photometry_figure": {"path": str(PHOTOMETRY_FIGURE), "sha256": sha256_file(PHOTOMETRY_FIGURE)},
    "vae_figure": {"path": str(VAE_FIGURE), "sha256": sha256_file(VAE_FIGURE)},
    "all_thresholds_pass": all_thresholds_pass,
    "qualification_inventory_sha256": qualification_inventory_sha256,
    "qualification_record_identities_sha256": qualification_inventory_evidence["sorted_record_identities_sha256"],
    "complete_qualification_inventory_used": True,
    "performance_based_selection_used": False,
    "gate01_continuity_only": True,
    "canonical_latent_bank_built": False,
    "descriptor_coupling_enabled": False,
}
review_outputs["manifest_sha256"] = sha256_json(review_outputs)
REVIEW_MANIFEST = output_root / "stage2_photometry_variant_a_review_manifest.json"
if REVIEW_MANIFEST.exists():
    if not ALLOW_EXACT_RESUME or json.loads(REVIEW_MANIFEST.read_text(encoding="utf-8")) != review_outputs:
        raise FileExistsError("Refusing to replace a non-identical review manifest.")
else:
    write_json_atomic(REVIEW_MANIFEST, review_outputs)
print(json.dumps({"review_manifest": str(REVIEW_MANIFEST), **review_outputs}, indent=2, sort_keys=True))
if not all_thresholds_pass:
    raise RuntimeError(
        f"FAIL CLOSED: Variant-A qualification failed {qualification['failure_classification']}. "
        "Do not build a latent bank or proceed to Stage 2."
    )
print("PASS: qualification evidence is ready for external review. This notebook intentionally stops without building a latent bank.")

## Final intentional stop

Return the JSON, Markdown, two compact figures, and review manifest for external review. Even after a pass, this notebook does not construct canonical artifacts, load `P` travellers, train a model, modify Variant-A arithmetic, or authorize descriptor coupling.